<a href="https://colab.research.google.com/github/russellheines/colab/blob/main/Conv2d_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST Digit Classifier with PyTorch

This notebook trains a Convolutional Neural Network (CNN) on the MNIST dataset.

### Pipeline Overview:
* **Framework:** PyTorch
* **Dataset:** MNIST Handwritten Digits (60,000 train / 10,000 test)
* **Architecture:** 2-Layer Convolutional Feature Extractor + 1 Linear Classifier
* **Output:** Trained model exported directly to universal ONNX format

### What does "MNIST" stand for?
The name stands for **Modified National Institute of Standards and Technology**.
* **NIST:** The original, raw handwriting samples were collected by a U.S. government agency (NIST) from high school students and Census Bureau employees.
* **Modified (M):** In the 1990s, AI pioneers Yann LeCun and his team *modified* the messy original data by mixing the writer groups together, antialiasing the pixel lines, and perfectly centering each number. This cleanup turned it into the pristine, gold-standard benchmark used across the AI world today.

### Dataset Details:
* **Image Specifications:** Each example is a standardized **28×28 pixel** grid (totaling 784 individual pixel inputs).
* **Color Channels:** Every image is **grayscale** (a single 1-channel matrix), where values range from 0 (pure black background) to 255 (pure white brush strokes).
* **Dataset Volume:** 70,000 total handwritten digits (60,000 for training, 10,000 for final test evaluation).

### Hardware Note: Why This Trains Easily on a CPU
While modern deep learning typically demands a heavy GPU, this complete MNIST model can be fully trained on a standard laptop CPU in **under a minute** due to its minimal computational workload:
* **Low Pixel Volume:** At only 784 pixels per image, the matrix math is tiny compared to modern multi-megapixel photos.
* **Minimal Memory Depth:** Grayscale data requires only 1 channel, meaning the computer's CPU memory cache never gets flooded.
* **Compact Architecture:** With only 4 total layers and around 421,000 parameters, a standard multi-core processor can easily handle the calculations sequentially without bottlenecking.


In [ ]:
import torch

# Check if a graphics card (GPU) is available in Colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transform each image into a 3D tensor with shape (1, 28, 28), converting grayscale values
# to floats and normalizing using a pre-calculated mean and standard deviation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,))
])

# Download the training and testing digit data
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Group images into small batches of 64 so we don't run out of memory
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.57MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.52MB/s]


In [ ]:
# Print the shape of the training data and the first transformed image
image, label = train_dataset[0]

print(f"""=== Training Dataset ===
Dataset Type:          {type(train_dataset)}
Data Type:             {type(train_dataset.data)}
Data Shape:            {train_dataset.data.shape}
Labels Type:           {type(train_dataset.targets)}
Labels Shape:          {train_dataset.targets.shape}

=== Training Item (Index 0) ===
Item Type:             {type(train_dataset[0])}
Image Type:            {type(image)}
Image Shape:           {image.shape}
Label Type:            {type(label)}
Label:                 {label}""")


=== Training Dataset ===
Dataset Type:          <class 'torchvision.datasets.mnist.MNIST'>
Data Type:             <class 'torch.Tensor'>
Data Shape:            torch.Size([60000, 28, 28])
Labels Type:           <class 'torch.Tensor'>
Labels Shape:          torch.Size([60000])

=== Training Item (Index 0) ===
Item Type:             <class 'tuple'>
Image Type:            <class 'torch.Tensor'>
Image Shape:           torch.Size([1, 28, 28])
Label Type:            <class 'int'>
Label:                 5


In [ ]:
import torch.nn as nn

class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        # Feature extractors
        self.features = nn.Sequential(

            # The first layer has 1 input channel (grayscale values) and 32 output channels,
            # one for each feature being extracted (lines, curves, etc.)
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # The second layer takes those 32 basic features and combines them into
            # 64 deeper, more complex feature shapes (loops, intersections, etc.)
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Linear classifier
        self.classifier = nn.Sequential(

            # Flatten the 2D shapes into a 1D vectors
            nn.Flatten(),

            # Connect features to 128 hidden neurons
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),

            # Output 10 scores (one for each digit 0-9)
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate using standard naming
model = ConvNet().to(device)

# Print the layer blueprint
print(model)

# Calculate total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters in Network: {total_params:,}")
print(f"Trainable Parameters       : {trainable_params:,}")


ConvNet(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)
Total Parameters in Network: 421,642
Trainable Parameters       : 421,642


In [ ]:
import torch.optim as optim

# Loss Function: Calculates a numeric penalty score for incorrect guesses
criterion = nn.CrossEntropyLoss()

# Optimizer: The engine that tweaks internal weights to fix mistakes
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Set the model to training mode
model.train()

for batch_idx, (images, labels) in enumerate(train_loader):
    # Move this specific batch of data to the GPU
    images, labels = images.to(device), labels.to(device)

    # Reset gradients from the last turn
    optimizer.zero_grad()

    # Forward pass: Let the model guess
    outputs = model(images)

    # Calculate the error penalty
    loss = criterion(outputs, labels)

    # Backward pass: Trace the error backwards through the network layers
    loss.backward()

    # Step: Adjust the internal weights slightly to improve next time
    optimizer.step()

print("Finished training batch!")

Finished training batch!


In [ ]:
# Set the model to evaluation mode
model.eval()

test_loss = 0
correct = 0

# Turn off gradient calculation to save memory and make it super fast
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Get the network's predictions
        outputs = model(images)

        # Accumulate the total loss penalty on test data
        test_loss += criterion(outputs, labels).item()

        # Find the highest score index (this is the network's final digit guess)
        predictions = outputs.argmax(dim=1, keepdim=True)

        # Count how many guesses perfectly match the true labels
        correct += predictions.eq(labels.view_as(predictions)).sum().item()

# Calculate final performance percentages
avg_test_loss = test_loss / len(test_loader)
accuracy = 100. * correct / len(test_loader.dataset)

print(f"=== Final Test Results ===")
print(f"Average Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy    : {accuracy:.2f}% ({correct}/{len(test_loader.dataset)} correct)")

=== Final Test Results ===
Average Test Loss: 0.0483
Test Accuracy    : 98.40% (9840/10000 correct)


In [ ]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 59.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [onnxscript]


In [ ]:
import torch.onnx

# Set the model to evaluation mode
model.eval()

# Create a dummy input tensor matching the shape the network expects
# Shape: (Batch Size=1, Channels=1, Width=28, Height=28)
dummy_input = torch.randn(1, 1, 28, 28).to(device)

# Specify the filename
onnx_filename = "mnist_convnet.onnx"

print(f"Tracing network and exporting model to {onnx_filename}...")

# Run the export engine
torch.onnx.export(
    model.to(device),       # Move the model to the active device for the trace
    dummy_input,            # The dummy input tensor used to map out the layers
    onnx_filename,          # Where to save the file
    export_params=True,     # Store the trained parameter weights inside the file
    opset_version=11,       # Standard ONNX version compatibility
    do_constant_folding=True, # Optimizes the mathematical operators for speed
    input_names=['input'],   # Explicitly name the input layer for easy API calling
    output_names=['output'], # Explicitly name the output score layer
    dynamic_axes={'input' : {0 : 'batch_size'},    # Allow your production application
                  'output' : {0 : 'batch_size'}}   # to pass varying batch sizes later
)

print("=== ONNX Export Complete! ===")

Tracing network and exporting model to mnist_convnet.onnx...


/tmp/ipykernel_914/1440198645.py:16: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1116 04:35:54.105000 914 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ConvNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ConvNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/BaseConverter.h:66: adapter_lookup: Ass

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
=== ONNX Export Complete! ===
